# MAP — Multi-Model QLoRA + RRF Ensemble: Kubeflow Training Pipeline

Full pipeline: sequential QLoRA (4-bit) fine-tuning of multiple LLMs with Patience-based Early Stopping,
followed by Reciprocal Rank Fusion (RRF) ensemble evaluation on a held-out test set.

| Section | What it does |
|---|---|
| 1. Import & Runtime | Libraries, GPU settings, multi-model config |
| 2. Helper Functions | Text cleaning, prompt building, MAP@3, RRF |
| 3. Data Prep | 3-way stratified split — 80 / 10 / 10 (train / val / holdout) |
| 4. Multi-Model Training | Sequential QLoRA fine-tuning with `EarlyStoppingCallback`, VRAM flush between models |
| 5. RRF Ensemble Evaluation | Per-model beam-search inference on holdout set → RRF → final MAP@3 |

## 1. Import & Runtime

In [ ]:
from __future__ import annotations

import gc
import os
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    try:
        torch.backends.cuda.enable_flash_sdp(True)
        torch.backends.cuda.enable_mem_efficient_sdp(True)
        torch.backends.cuda.enable_math_sdp(True)
    except Exception:
        pass

# ── Multi-model list ──────────────────────────────────────────────────────────
# Add or remove model IDs here; all must share the same LoRA target-module names.
# The three models below are all decoder-only transformers with identical proj names.
MODEL_LIST = [
    "google/gemma-3-1b-it",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

# ── Paths & hyper-parameters ──────────────────────────────────────────────────
HF_TOKEN           = os.getenv("HF_TOKEN", "")
ROOT               = Path.cwd()
TRAIN_FILE         = ROOT / "train.csv"
MAX_SEQ_LENGTH     = int(os.getenv("MAX_SEQ_LENGTH", "1024"))
TRAIN_EPOCHS       = float(os.getenv("TRAIN_EPOCHS", "3"))
OUTPUT_ROOT        = Path(os.getenv("OUTPUT_ROOT", "./kubeflow_artifacts"))
FINAL_WEIGHTS_ROOT = OUTPUT_ROOT / "final_weights"   # one sub-dir per model
CACHE_DIR          = OUTPUT_ROOT / "hf_cache"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_LIST        : {MODEL_LIST}")
print(f"FINAL_WEIGHTS_ROOT: {FINAL_WEIGHTS_ROOT}")
print(f"CUDA available    : {torch.cuda.is_available()}")

## 2. Helper Functions

In [ ]:
# ── Text utilities ────────────────────────────────────────────────────────────

def normalize_text(value: object, default: str = "") -> str:
    if value is None:
        return default
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return default
    return text


def make_target_label(category: object, misconception: object) -> str:
    category_text    = normalize_text(category,    default="Unknown")
    misconception_text = normalize_text(misconception, default="NA") or "NA"
    return f"{category_text}:{misconception_text}"


def build_user_prompt(
    question: object, correct_answer: object, student_explanation: object
) -> str:
    return (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "Return only the label.\n\n"
        f"Question: {normalize_text(question)}\n"
        f"Correct answer: {normalize_text(correct_answer)}\n"
        f"Student explanation: {normalize_text(student_explanation)}\n"
    )


# ── Dataset formatting — model-agnostic via apply_chat_template ───────────────

def make_dataset_for_model(df: pd.DataFrame, tokenizer) -> Dataset:
    """Convert a DataFrame to a Dataset using the tokenizer's own chat template."""
    def _format(example):
        messages = [
            {
                "role": "user",
                "content": build_user_prompt(
                    example["QuestionText"],
                    example["MC_Answer"],
                    example["StudentExplanation"],
                ),
            },
            {"role": "assistant", "content": example["target_text"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        return {"text": text, "label_text": example["target_text"]}

    ds = Dataset.from_pandas(df.reset_index(drop=True), preserve_index=False)
    return ds.map(_format, remove_columns=ds.column_names)


def build_inference_prompt(row, tokenizer) -> str:
    """Build the prompt prefix for inference (no assistant turn)."""
    messages = [
        {
            "role": "user",
            "content": build_user_prompt(
                row.QuestionText, row.MC_Answer, row.StudentExplanation
            ),
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# ── Model-ID → safe filesystem slug ──────────────────────────────────────────

def get_model_slug(model_id: str) -> str:
    """'org/model-name-v2' → 'org__model_name_v2'"""
    return model_id.replace("/", "__").replace("-", "_").replace(".", "_")


# ── MAP@3 ─────────────────────────────────────────────────────────────────────

def calculate_row_map3(true_label: str, predicted_list, k: int = 3) -> float:
    true_clean = normalize_text(true_label)
    seen: list[str] = []
    for p in predicted_list:
        p_clean = normalize_text(p)
        if p_clean and p_clean not in seen:
            seen.append(p_clean)
    top_k_unique = seen[:k]
    if true_clean in top_k_unique:
        return 1.0 / (top_k_unique.index(true_clean) + 1)
    return 0.0


def calculate_map3(
    true_labels, predicted_lists, k: int = 3
) -> float:
    if not true_labels:
        return 0.0
    scores = [
        calculate_row_map3(t, p, k=k)
        for t, p in zip(true_labels, predicted_lists)
    ]
    return float(sum(scores) / len(scores))


# ── Reciprocal Rank Fusion (RRF) ──────────────────────────────────────────────

def rrf_fusion(
    model_predictions: list[list[list[str]]],
    top_k: int = 3,
    k_rrf: int = 60,
    fallback_labels: list[str] | None = None,
) -> list[list[str]]:
    """
    Fuse ranked candidate lists from multiple models via RRF.

    score(label) += 1.0 / (k_rrf + rank + 1)   for each model that ranked it

    Args:
        model_predictions : [num_models][num_rows][num_candidates]
        top_k             : labels to return per row
        k_rrf             : smoothing constant (60 is standard)
        fallback_labels   : ordered fallback list used when <top_k unique candidates

    Returns:
        fused_results : [num_rows][top_k]
    """
    num_rows = len(model_predictions[0])
    fused_results: list[list[str]] = []

    for row_idx in range(num_rows):
        scores: dict[str, float] = defaultdict(float)

        for model_preds in model_predictions:
            # De-duplicate while preserving order (first occurrence keeps rank)
            seen: list[str] = []
            for cand in model_preds[row_idx]:
                clean = normalize_text(cand)
                if clean and clean not in seen:
                    seen.append(clean)
            for rank, label in enumerate(seen):
                scores[label] += 1.0 / (k_rrf + rank + 1)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top_labels = [label for label, _ in ranked[:top_k]]

        # Pad to top_k with the most-common training labels
        if fallback_labels:
            for fb in fallback_labels:
                if len(top_labels) >= top_k:
                    break
                if fb not in top_labels:
                    top_labels.append(fb)

        fused_results.append(top_labels)

    return fused_results


# ── Label-only accuracy for early stopping ────────────────────────────────────

def preprocess_logits_for_metrics(logits, labels):
    """Convert full logit tensor to argmax predictions to save GPU memory."""
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def compute_label_accuracy(eval_preds):
    """
    Token-level accuracy computed only on label (completion) tokens.

    Prompt tokens are masked to -100 by DataCollatorForCompletionOnlyLM,
    so mask = (shift_labels != -100) isolates exactly the response tokens.
    """
    predictions, labels = eval_preds   # both shape [N, seq_len]
    # Causal LM: prediction at position i targets the token at position i+1
    shift_preds  = predictions[:, :-1]
    shift_labels = labels[:, 1:]
    mask  = shift_labels != -100       # True only for completion tokens
    total = int(mask.sum())
    if total == 0:
        return {"label_accuracy": 0.0}
    correct = int(((shift_preds == shift_labels) & mask).sum())
    return {"label_accuracy": correct / total}

## 3. Data Prep — Stratified 80 / 10 / 10 Split

* **Train set (80%)** — used for model fine-tuning.
* **Validation set (10%)** — used only for loss monitoring and Early Stopping; never seen by the final evaluation.
* **Holdout set (10%)** — held out completely during training; used exclusively for the final RRF MAP@3 evaluation.

In [ ]:
df = pd.read_csv(TRAIN_FILE)

required_columns = {"QuestionText", "MC_Answer", "StudentExplanation", "Category", "Misconception"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"train.csv is missing columns: {sorted(missing)}")

df = df.copy()
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    df[col] = df[col].fillna("")
df["target_text"] = df.apply(
    lambda r: make_target_label(r["Category"], r["Misconception"]), axis=1
)

# Fallback labels ordered by frequency (most common first)
label_counts   = Counter(df["target_text"].tolist())
fallback_labels = [label for label, _ in label_counts.most_common()]

# ── Step 1: carve out 10 % holdout ───────────────────────────────────────────
train_val_df, holdout_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    stratify=df["Category"].astype(str),
)

# ── Step 2: split remaining 90 % into 80 % train / 10 % val ─────────────────
# 10 % of total = 10/90 ≈ 11.1 % of the 90 % train_val block
train_df, val_df = train_test_split(
    train_val_df,
    test_size=round(1 / 9, 6),
    random_state=42,
    stratify=train_val_df["Category"].astype(str),
)

total = len(df)
print(f"Total rows  : {total}")
print(f"Train rows  : {len(train_df)}  ({100*len(train_df)/total:.1f}%)")
print(f"Val rows    : {len(val_df)}   ({100*len(val_df)/total:.1f}%)")
print(f"Holdout rows: {len(holdout_df)}  ({100*len(holdout_df)/total:.1f}%)")
print(f"Top-3 fallback labels: {fallback_labels[:3]}")

## 4. Multi-Model Training with Patience Early Stopping

For each model in `MODEL_LIST`:
1. Download base weights.
2. Format datasets with the model's own chat template.
3. Load model in 4-bit NF4 + apply LoRA adapters.
4. Train with `EarlyStoppingCallback(patience=3)` — stops if `eval_loss` does not improve for 3 consecutive eval checkpoints.
5. Save adapter + tokenizer to `final_weights/<slug>/`.
6. **Completely purge model, trainer, tokenizer from VRAM** before the next iteration.

In [ ]:
# ── Shared QLoRA quantisation config ─────────────────────────────────────────
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

# ── Shared LoRA config ────────────────────────────────────────────────────────
# q/k/v/o/gate/up/down_proj names are identical across Gemma-3, Llama-3.2, Qwen2.5.
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# ── Sequential training loop ──────────────────────────────────────────────────
trained_model_slugs: list[str] = []

for model_id in MODEL_LIST:
    slug        = get_model_slug(model_id)
    adapter_dir = FINAL_WEIGHTS_ROOT / slug
    ckpt_dir    = OUTPUT_ROOT / "checkpoints" / slug

    print(f"\n{'='*70}")
    print(f"[TRAIN] {model_id}")
    print(f"        adapter → {adapter_dir}")
    print(f"{'='*70}")

    # 1. Download base weights (skips if already cached)
    base_local_dir = snapshot_download(
        repo_id=model_id,
        token=HF_TOKEN or None,
        cache_dir=str(CACHE_DIR),
        local_dir_use_symlinks=False,
    )

    # 2. Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        base_local_dir, trust_remote_code=True, use_fast=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # 3. Detect response template from this model's chat template and build a
    #    DataCollatorForCompletionOnlyLM so that prompt tokens are masked to -100.
    #    This restricts both the training loss AND the label_accuracy metric to
    #    the assistant's response tokens only (the "Category:Misconception" part).
    _with_gen    = tokenizer.apply_chat_template(
        [{"role": "user", "content": "X"}], tokenize=False, add_generation_prompt=True
    )
    _without_gen = tokenizer.apply_chat_template(
        [{"role": "user", "content": "X"}], tokenize=False, add_generation_prompt=False
    )
    response_template_str = _with_gen[len(_without_gen):]
    data_collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template_str,
        tokenizer=tokenizer,
        mlm=False,
    )
    print(f"[TRAIN] response_template = {repr(response_template_str)}")

    # 4. Format datasets with this model's chat template
    train_ds = make_dataset_for_model(train_df, tokenizer)
    val_ds   = make_dataset_for_model(val_df,   tokenizer)
    print(f"[TRAIN] train_ds={len(train_ds)} rows  val_ds={len(val_ds)} rows")

    # 5. Load base model in 4-bit NF4
    # NOTE: `dtype` replaces the deprecated `torch_dtype` in transformers >=4.45
    model = AutoModelForCausalLM.from_pretrained(
        base_local_dir,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=compute_dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.requires_grad_(False)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # 6. SFT config — eval every 50 steps, early stopping patience = 3
    # NOTE: `dataset_text_field` moved from SFTTrainer to SFTConfig in TRL 0.9+/0.24.x
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    sft_config = SFTConfig(
        output_dir=str(ckpt_dir),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=TRAIN_EPOCHS,
        logging_steps=20,
        # Early-stopping requires steps-based evaluation
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="label_accuracy",   # monitor label-only token accuracy
        greater_is_better=True,                   # higher accuracy = better
        report_to="none",
        max_length=MAX_SEQ_LENGTH,
        bf16=(compute_dtype == torch.bfloat16),
        fp16=(compute_dtype == torch.float16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        # completion_only_loss is handled via DataCollatorForCompletionOnlyLM above
        completion_only_loss=False,
        packing=False,
        dataset_text_field="text",
    )

    # 7. Trainer with EarlyStoppingCallback (patience=3 means 3 consecutive
    #    eval checkpoints with no label_accuracy improvement → training stops)
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_label_accuracy,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    print(f"[TRAIN] Finished. Best checkpoint: {trainer.state.best_model_checkpoint}")

    # 8. Save best adapter + tokenizer
    adapter_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"[TRAIN] Adapter saved → {adapter_dir}")
    trained_model_slugs.append(slug)

    # 9. !! VRAM CLEANUP — must del all references before next model loads !!
    del trainer, model, tokenizer, train_ds, val_ds, data_collator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[TRAIN] VRAM cleared after {model_id}")

print(f"\n✓ All models trained: {trained_model_slugs}")

## 5. RRF Ensemble Evaluation — Holdout MAP@3

For each trained model:
1. Load the base model + LoRA adapter (`PeftModel.from_pretrained`).
2. Run beam-search inference (4 beams, 4 return sequences) on every holdout row.
3. Immediately delete the model and flush VRAM.

After all models finish, apply **Reciprocal Rank Fusion** across models' candidate lists and compute the final MAP@3.

In [ ]:
# Collect per-model predictions: shape [num_models][num_holdout_rows][num_candidates]
all_model_predictions: list[list[list[str]]] = []

for model_id in MODEL_LIST:
    slug        = get_model_slug(model_id)
    adapter_dir = FINAL_WEIGHTS_ROOT / slug

    if not adapter_dir.exists():
        print(f"[EVAL] SKIP {model_id}: adapter not found at {adapter_dir}")
        continue

    print(f"\n{'='*70}")
    print(f"[EVAL] {model_id}")
    print(f"{'='*70}")

    # ── Load tokenizer from the saved adapter directory ───────────────────────
    tokenizer = AutoTokenizer.from_pretrained(
        str(adapter_dir), trust_remote_code=True, use_fast=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── Load base model in 4-bit, then attach LoRA weights ───────────────────
    # NOTE: `dtype` replaces the deprecated `torch_dtype` in transformers >=4.45
    eval_bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base_local_dir = snapshot_download(
        repo_id=model_id,
        token=HF_TOKEN or None,
        cache_dir=str(CACHE_DIR),
        local_dir_use_symlinks=False,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        base_local_dir,
        quantization_config=eval_bnb,
        device_map="auto",
        dtype=compute_dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, str(adapter_dir))
    model.eval()
    model.config.use_cache = True

    device = next(model.parameters()).device
    row_predictions: list[list[str]] = []

    # ── Row-by-row beam-search on the entire holdout set ─────────────────────
    for i, row in enumerate(holdout_df.itertuples(index=False)):
        full_prompt = build_inference_prompt(row, tokenizer)
        inputs = tokenizer(
            full_prompt, return_tensors="pt", add_special_tokens=False
        )
        inputs     = {k: v.to(device) for k, v in inputs.items()}
        prompt_len = inputs["input_ids"].shape[-1]

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                num_beams=4,
                num_return_sequences=4,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                early_stopping=True,
            )

        candidates: list[str] = []
        for seq in outputs:
            decoded    = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True).strip()
            first_line = decoded.splitlines()[0].strip() if decoded else ""
            if first_line:
                candidates.append(first_line)
        row_predictions.append(candidates)

        if (i + 1) % 50 == 0:
            print(f"[EVAL] {model_id}: {i+1}/{len(holdout_df)} rows done")

    all_model_predictions.append(row_predictions)
    print(f"[EVAL] {model_id}: inference complete ({len(row_predictions)} rows)")

    # ── VRAM cleanup before next model ────────────────────────────────────────
    del model, base_model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[EVAL] VRAM cleared after {model_id}")

# ── RRF fusion across all collected predictions ───────────────────────────────
if not all_model_predictions:
    raise RuntimeError("No model predictions collected — check that adapters exist in FINAL_WEIGHTS_ROOT.")

print(f"\n[RRF] Fusing {len(all_model_predictions)} model(s) over {len(holdout_df)} holdout rows...")
fused_predictions = rrf_fusion(
    model_predictions=all_model_predictions,
    top_k=3,
    k_rrf=60,
    fallback_labels=fallback_labels,
)

# ── Final MAP@3 on holdout set ────────────────────────────────────────────────
holdout_true_labels = holdout_df["target_text"].tolist()
final_map3 = calculate_map3(holdout_true_labels, fused_predictions, k=3)

print(f"\n{'='*70}")
print(f"[RESULT] RRF Ensemble MAP@3 on Holdout Set : {final_map3:.6f}")
print(f"[RESULT] Models fused                      : {len(all_model_predictions)}")
print(f"[RESULT] Holdout rows evaluated             : {len(holdout_true_labels)}")
print(f"{'='*70}")